<a href="https://colab.research.google.com/github/Rubal-code/langgraph/blob/main/11_persistance.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [11]:
!pip install langchain langgraph langchain_groq

In [12]:
from langgraph.graph import StateGraph,START,END
from langchain_groq import ChatGroq
from langgraph.checkpoint.memory import InMemorySaver
from typing import TypedDict

In [14]:
llm=ChatGroq(model='llama-3.3-70b-versatile')

In [15]:
class JokeState(TypedDict):

    topic: str
    joke: str
    explanation: str

In [16]:
def generate_joke(state: JokeState):

    prompt = f'generate a joke on the topic {state["topic"]}'
    response = llm.invoke(prompt).content

    return {'joke': response}

In [17]:
def generate_explanation(state: JokeState):

    prompt = f'write an explanation for the joke - {state["joke"]}'
    response = llm.invoke(prompt).content

    return {'explanation': response}

In [18]:
graph = StateGraph(JokeState)

graph.add_node('generate_joke', generate_joke)
graph.add_node('generate_explanation', generate_explanation)

graph.add_edge(START, 'generate_joke')
graph.add_edge('generate_joke', 'generate_explanation')
graph.add_edge('generate_explanation', END)

checkpointer = InMemorySaver()

workflow = graph.compile(checkpointer=checkpointer)

In [19]:
config1 = {"configurable": {"thread_id": "1"}}
workflow.invoke({'topic':'pizza'}, config=config1)

{'topic': 'pizza',
 'joke': 'Why was the pizza in a bad mood?\n\nBecause it was feeling a little crusty!',
 'explanation': 'A classic play on words. The joke "Why was the pizza in a bad mood? Because it was feeling a little crusty!" is a clever pun that relies on a double meaning of the word "crusty".\n\nIn one sense, "crusty" refers to the outer layer of a pizza, which is typically crispy and golden brown. However, "crusty" can also be used to describe someone\'s personality or mood, implying that they are irritable, gruff, or unpleasant.\n\nThe joke sets up the expectation that the pizza is in a bad mood, and the punchline subverts this expectation by using the word "crusty" in a way that references both the pizza\'s physical characteristics and its emotional state. The humor comes from the unexpected twist on the word\'s meaning, creating a clever and amusing connection between the setup and the punchline.\n\nIn essence, the joke is saying that the pizza is in a bad mood because it\

In [20]:
config2 = {"configurable": {"thread_id": "2"}}
workflow.invoke({'topic':'pasta'}, config=config2)

{'topic': 'pasta',
 'joke': 'Why did the spaghetti refuse to get married?\n\nBecause it was afraid of getting tangled up in a lifelong commitment!',
 'explanation': 'A clever play on words. This joke is humorous because it takes the common phrase "tangled up in a lifelong commitment" (which typically refers to the complexities and responsibilities that come with marriage) and gives it a literal twist. \n\nIn this joke, the spaghetti (a type of long, thin, and flexible pasta) is personified, and its fear of getting "tangled up" is a pun on the fact that spaghetti is prone to becoming knotted and entwined when cooked or handled. The phrase "tangled up" is usually used figuratively to describe a complicated situation, but in this case, it\'s also a literal concern for the spaghetti, as it could actually become physically tangled.\n\nThe humor comes from the unexpected shift in meaning, as the listener is initially expecting a typical reason for not wanting to get married (e.g., fear of re

In [21]:
workflow.get_state(config1)

StateSnapshot(values={'topic': 'pizza', 'joke': 'Why was the pizza in a bad mood?\n\nBecause it was feeling a little crusty!', 'explanation': 'A classic play on words. The joke "Why was the pizza in a bad mood? Because it was feeling a little crusty!" is a clever pun that relies on a double meaning of the word "crusty".\n\nIn one sense, "crusty" refers to the outer layer of a pizza, which is typically crispy and golden brown. However, "crusty" can also be used to describe someone\'s personality or mood, implying that they are irritable, gruff, or unpleasant.\n\nThe joke sets up the expectation that the pizza is in a bad mood, and the punchline subverts this expectation by using the word "crusty" in a way that references both the pizza\'s physical characteristics and its emotional state. The humor comes from the unexpected twist on the word\'s meaning, creating a clever and amusing connection between the setup and the punchline.\n\nIn essence, the joke is saying that the pizza is in a b

In [22]:
list[workflow.get_state(config1)]

list[{'topic': 'pizza', 'joke': 'Why was the pizza in a bad mood?\n\nBecause it was feeling a little crusty!', 'explanation': 'A classic play on words. The joke "Why was the pizza in a bad mood? Because it was feeling a little crusty!" is a clever pun that relies on a double meaning of the word "crusty".\n\nIn one sense, "crusty" refers to the outer layer of a pizza, which is typically crispy and golden brown. However, "crusty" can also be used to describe someone\'s personality or mood, implying that they are irritable, gruff, or unpleasant.\n\nThe joke sets up the expectation that the pizza is in a bad mood, and the punchline subverts this expectation by using the word "crusty" in a way that references both the pizza\'s physical characteristics and its emotional state. The humor comes from the unexpected twist on the word\'s meaning, creating a clever and amusing connection between the setup and the punchline.\n\nIn essence, the joke is saying that the pizza is in a bad mood because 

# time travel


In [24]:
workflow.get_state({"configurable": {"thread_id": "1", "checkpoint_id": "1f187152-e5d1-6ff4-8001-64de89719d8a"}})

StateSnapshot(values={'topic': 'pizza', 'joke': 'Why was the pizza in a bad mood?\n\nBecause it was feeling a little crusty!'}, next=('generate_explanation',), config={'configurable': {'thread_id': '1', 'checkpoint_id': '1f187152-e5d1-6ff4-8001-64de89719d8a'}}, metadata={'source': 'loop', 'step': 1, 'parents': {}}, created_at='2026-07-24T04:07:26.711362+00:00', parent_config={'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'checkpoint_id': '1f187152-e2f8-659e-8000-3ce34859182a'}}, tasks=(PregelTask(id='897cff0d-1bdd-cbe9-15ca-721c583cca56', name='generate_explanation', path=('__pregel_pull', 'generate_explanation'), error=None, interrupts=(), state=None, result={'explanation': 'A classic play on words. The joke "Why was the pizza in a bad mood? Because it was feeling a little crusty!" is a clever pun that relies on a double meaning of the word "crusty".\n\nIn one sense, "crusty" refers to the outer layer of a pizza, which is typically crispy and golden brown. However, "crusty" 

In [25]:
workflow.invoke(None, {"configurable": {"thread_id": "1", "checkpoint_id": "1f187152-e5d1-6ff4-8001-64de89719d8a"}})

{'topic': 'pizza',
 'joke': 'Why was the pizza in a bad mood?\n\nBecause it was feeling a little crusty!',
 'explanation': 'A classic play on words. The joke relies on a pun to create humor. Here\'s a breakdown of how it works:\n\nThe question sets up the expectation that the pizza is in a bad mood, which is a common human emotion. The punchline "it was feeling a little crusty" has a double meaning:\n\n1. **Crusty** can describe the outer layer of a pizza, which is crispy and golden brown. This is a literal characteristic of a pizza.\n2. **Crusty** can also be an idiomatic expression meaning grumpy, irritable, or having a sour disposition. This is a figurative way to describe someone\'s (or in this case, a pizza\'s) mood.\n\nThe humor comes from the unexpected twist on the word "crusty". The listener is initially expecting a reason related to the pizza\'s emotions, but instead, the joke uses the wordplay to connect the pizza\'s literal crust to its emotional state. The result is a ligh

In [26]:
list(workflow.get_state_history(config1))

[StateSnapshot(values={'topic': 'pizza', 'joke': 'Why was the pizza in a bad mood?\n\nBecause it was feeling a little crusty!', 'explanation': 'A classic play on words. The joke relies on a pun to create humor. Here\'s a breakdown of how it works:\n\nThe question sets up the expectation that the pizza is in a bad mood, which is a common human emotion. The punchline "it was feeling a little crusty" has a double meaning:\n\n1. **Crusty** can describe the outer layer of a pizza, which is crispy and golden brown. This is a literal characteristic of a pizza.\n2. **Crusty** can also be an idiomatic expression meaning grumpy, irritable, or having a sour disposition. This is a figurative way to describe someone\'s (or in this case, a pizza\'s) mood.\n\nThe humor comes from the unexpected twist on the word "crusty". The listener is initially expecting a reason related to the pizza\'s emotions, but instead, the joke uses the wordplay to connect the pizza\'s literal crust to its emotional state. 

# updating task

In [29]:
workflow.update_state({"configurable": {"thread_id": "1", "checkpoint_id": "1f187152-e5d1-6ff4-8001-64de89719d8a", "checkpoint_ns": ""}}, {'topic':'samosa'})

{'configurable': {'thread_id': '1',
  'checkpoint_ns': '',
  'checkpoint_id': '1f187165-9238-62a8-8002-506078a04f0e'}}

In [30]:
list(workflow.get_state_history(config1))

[StateSnapshot(values={'topic': 'samosa', 'joke': 'Why was the pizza in a bad mood?\n\nBecause it was feeling a little crusty!'}, next=('generate_explanation',), config={'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'checkpoint_id': '1f187165-9238-62a8-8002-506078a04f0e'}}, metadata={'source': 'update', 'step': 2, 'parents': {}}, created_at='2026-07-24T04:15:47.972544+00:00', parent_config={'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'checkpoint_id': '1f187152-e5d1-6ff4-8001-64de89719d8a'}}, tasks=(PregelTask(id='3d22ed76-2e3b-4054-ff5a-3a1d0f16307a', name='generate_explanation', path=('__pregel_pull', 'generate_explanation'), error=None, interrupts=(), state=None, result=None),), interrupts=()),
 StateSnapshot(values={'topic': 'samosa', 'joke': 'Why was the pizza in a bad mood?\n\nBecause it was feeling a little crusty!', 'explanation': 'A classic play on words. The joke "Why was the pizza in a bad mood? Because it was feeling a little crusty!" is a clever pun th

In [31]:
workflow.invoke(None,{"configurable":{"thread_id":"1","checkpoint":"1f187165-9238-62a8-8002-506078a04f0e"}})

{'topic': 'samosa',
 'joke': 'Why was the pizza in a bad mood?\n\nBecause it was feeling a little crusty!',
 'explanation': 'A deliciously cheesy joke. Let\'s break it down:\n\nThe joke is a play on words, using a common phrase "feeling a little crusty" and giving it a double meaning. In everyday language, "feeling crusty" is an idiomatic expression that means being irritable, grumpy, or having a bad temper. However, in the context of a pizza, "crusty" has a literal meaning, referring to the crust of the pizza, which is the outer layer of the bread.\n\nThe humor comes from the unexpected twist on the word "crusty". The setup "Why was the pizza in a bad mood?" primes the listener to expect a reason related to the pizza\'s emotions or circumstances. But the punchline subverts this expectation by using the word "crusty" in a way that is both a clever pun and a literal reference to the pizza\'s composition.\n\nIn essence, the joke is saying that the pizza is in a bad mood because it\'s fee